# 06 — Étude Complète QMKL : Bayesian Optimization, Scaling, Multi-Dataset

Étude approfondie du QMKL avec :
- **20 kernels** (bibliothèque étendue)
- **6 méthodes MKL** : Single-Best, Average, Centered, SDP, Projection, **BO-MKQSVM**
- **3 datasets** : German Credit, Bank Marketing, Synthetic
- **Scaling jusqu'à 12 qubits**
- **14 figures** d'analyse

### Table des figures
| # | Description |
|---|-------------|
| F1 | Grille de heatmaps de kernels sélectionnés |
| F2 | Alignement kernel-target (ranking) |
| F3 | Comparaison 6 méthodes × 3 datasets (barres groupées) |
| F4 | Radar chart multi-métriques |
| F5 | Convergence BO par dataset |
| F6 | Sensibilité BO : n_calls vs AUC |
| F7 | Poids BO vs Centered (scatter) |
| F8 | Scaling n_qubits avec CI95 |
| F9 | Heatmap concentration (n_qubits × kernel) |
| F10 | Heatmap poids (méthode × kernel) |
| F11 | Violin plots stabilité des poids |
| F12 | Concentration vs performance (scatter) |
| F13 | Fidelity vs Projected |
| F14 | Panel de synthèse 2×3 |

In [ ]:
import sys, os, time, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

from data.loaders import load_dataset
from src.preprocessing import QuantumScaler, FeatureReducer
from src.kernels import build_feature_map, build_quantum_kernel, compute_kernel_matrix
from src.kernels.kernel_matrix import compute_kernel_matrix_parallel, kernel_statistics, cache_info
from src.kernels.feature_maps import get_extended_feature_map_library
from src.mkl.alignment import centered_alignment, sdp_alignment, projection_alignment
from src.mkl.bayesian_optimizer import BayesianKernelOptimizer
from src.evaluation.visualization import (
    plot_method_comparison_grouped, plot_bo_convergence, plot_weight_heatmap,
    plot_scaling_curve, plot_concentration_scatter, plot_radar_chart,
    _kernel_off_diagonal_stats,
)

os.makedirs('../results/06', exist_ok=True)
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
np.random.seed(SEED)

info = cache_info()
print(f'Cache : {info["files"]} fichiers ({info["total_size_mb"]:.1f} MB)')

## §1 — Chargement des 3 datasets

In [ ]:
N_SAMPLES = 250
N_QUBITS  = 6  # PCA default

DATASETS = {}
for name in ['german_credit', 'bank_marketing', 'synthetic']:
    X, y = load_dataset(name, n_samples=N_SAMPLES, random_state=SEED)
    reducer = FeatureReducer(n_components=N_QUBITS)
    scaler  = QuantumScaler(feature_range=(0, 2))
    X_proc = scaler.fit_transform(reducer.fit_transform(X))
    DATASETS[name] = {'X': X_proc, 'y': y, 'X_raw': X}
    print(f'{name:20s}: X={X_proc.shape}, classes={np.bincount(y)}')

## §2 — Bibliothèque étendue de 20 kernels + précalcul

In [ ]:
# Build 20 kernels
fm_lib = get_extended_feature_map_library(N_QUBITS)
KNAMES = list(fm_lib.keys())
print(f'{len(KNAMES)} feature maps : {KNAMES}\n')

# Precompute all kernel matrices per dataset (cached)
KERNELS = {}  # {dataset: list of K_full}
for ds_name, ds in DATASETS.items():
    print(f'Computing kernels for {ds_name}...')
    t0 = time.time()
    qk_list = [build_quantum_kernel(fm, kernel_type='fidelity') for fm in fm_lib.values()]
    K_list = compute_kernel_matrix_parallel(
        qk_list, ds['X'],
        kernel_names=[f'{ds_name}_fid_{n}' for n in KNAMES],
        n_jobs=-1
    )
    KERNELS[ds_name] = K_list
    print(f'  {len(K_list)} matrices in {time.time()-t0:.1f}s')

print(f'\nCache : {cache_info()["files"]} fichiers')

In [ ]:
# ── FIGURE 1 : Grille 3×3 de heatmaps de kernels sélectionnés ───────────────
sample_idx = [0, 3, 6, 8, 11, 14, 17, 19, 1]
fig, axes = plt.subplots(3, 3, figsize=(15, 13))
for ax, idx in zip(axes.flat, sample_idx):
    K = KERNELS['synthetic'][idx]
    sns.heatmap(K[:50, :50], cmap='viridis', ax=ax, cbar=False, square=True)
    ax.set_title(KNAMES[idx], fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Figure 1 — Échantillon de matrices kernel (50×50, dataset synthetic)', fontsize=14)
plt.tight_layout()
plt.savefig('../results/06/F01_kernel_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── FIGURE 2 : Alignement kernel-target ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (ds_name, ds) in zip(axes, DATASETS.items()):
    Ky = np.outer(ds['y'], ds['y']).astype(float)
    aligns = []
    for K in KERNELS[ds_name]:
        num = np.sum(K * Ky)
        denom = np.linalg.norm(K, 'fro') * np.linalg.norm(Ky, 'fro')
        aligns.append(num / denom if denom > 0 else 0)
    order = np.argsort(aligns)[::-1]
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(KNAMES)))
    ax.barh(range(len(KNAMES)), [aligns[i] for i in order],
            color=[colors[j] for j in range(len(KNAMES))], edgecolor='black', linewidth=0.3)
    ax.set_yticks(range(len(KNAMES)))
    ax.set_yticklabels([KNAMES[i] for i in order], fontsize=7)
    ax.set_xlabel('Alignement')
    ax.set_title(ds_name)
    ax.invert_yaxis()

plt.suptitle('Figure 2 — Alignement kernel-target par dataset (20 kernels)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/06/F02_alignment_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## §3 — Comparaison des 6 méthodes MKL

In [ ]:
def evaluate_method(K_list, y, weight_fn, n_folds=5, C=1.0, seed=42):
    """Evaluate a single MKL method via stratified K-fold CV.
    Returns dict with mean, std, and per-metric averages."""
    n = len(y)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    all_metrics = {'auc': [], 'acc': [], 'f1': [], 'prec': [], 'recall': []}
    all_weights = []

    for idx_tr, idx_te in skf.split(np.zeros(n), y):
        y_tr, y_te = y[idx_tr], y[idx_te]
        K_tr = [K[np.ix_(idx_tr, idx_tr)] for K in K_list]
        K_te = [K[np.ix_(idx_te, idx_tr)] for K in K_list]

        w = weight_fn(K_tr, y_tr)
        w = np.array(w)
        if w.sum() > 0: w = w / w.sum()
        else: w = np.ones(len(K_list)) / len(K_list)
        all_weights.append(w)

        Kc_tr = sum(wi * K for wi, K in zip(w, K_tr))
        Kc_te = sum(wi * K for wi, K in zip(w, K_te))
        ev = np.linalg.eigvalsh(Kc_tr)
        if ev.min() < 0:
            Kc_tr += (abs(ev.min()) + 1e-10) * np.eye(Kc_tr.shape[0])

        svm = SVC(kernel='precomputed', C=C, probability=True)
        svm.fit(Kc_tr, y_tr)
        y_pred  = svm.predict(Kc_te)
        y_proba = svm.predict_proba(Kc_te)[:, 1]

        all_metrics['auc'].append(roc_auc_score(y_te, y_proba))
        all_metrics['acc'].append(accuracy_score(y_te, y_pred))
        all_metrics['f1'].append(f1_score(y_te, y_pred, zero_division=0))
        all_metrics['prec'].append(precision_score(y_te, y_pred, zero_division=0))
        all_metrics['recall'].append(recall_score(y_te, y_pred, zero_division=0))

    return {
        'mean': float(np.mean(all_metrics['auc'])),
        'std':  float(np.std(all_metrics['auc'], ddof=1)),
        'scores': all_metrics['auc'],
        'all_metrics': {k: float(np.mean(v)) for k, v in all_metrics.items()},
        'mean_weights': np.mean(all_weights, axis=0),
    }


def weight_fn_single_best(K_list, y_tr):
    Ky = np.outer(y_tr, y_tr).astype(float)
    aligns = [np.sum(K * Ky) / (np.linalg.norm(K,'fro') * np.linalg.norm(Ky,'fro') + 1e-10)
              for K in K_list]
    w = np.zeros(len(K_list)); w[np.argmax(aligns)] = 1.0
    return w

def weight_fn_average(K_list, y_tr):
    return np.ones(len(K_list)) / len(K_list)

def weight_fn_centered(K_list, y_tr):
    return centered_alignment(K_list, np.outer(y_tr, y_tr).astype(float))

def weight_fn_sdp(K_list, y_tr):
    return sdp_alignment(K_list, np.outer(y_tr, y_tr).astype(float))

def weight_fn_projection(K_list, y_tr):
    return projection_alignment(K_list, np.outer(y_tr, y_tr).astype(float))

# BO needs special handling (not a simple weight_fn)
def evaluate_bo(K_list, y, n_folds=5, n_calls=50, seed=42):
    n = len(y)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    all_auc = []
    all_weights = []
    bo_histories = []

    for idx_tr, idx_te in skf.split(np.zeros(n), y):
        y_tr, y_te = y[idx_tr], y[idx_te]
        K_tr = [K[np.ix_(idx_tr, idx_tr)] for K in K_list]
        K_te = [K[np.ix_(idx_te, idx_tr)] for K in K_list]

        bo = BayesianKernelOptimizer(n_calls=n_calls, n_initial_points=10,
                                     random_state=seed)
        w, C = bo.optimize(K_tr, y_tr, scoring='accuracy')
        bo_histories.append(bo.get_convergence_history())
        all_weights.append(w)

        Kc_tr = sum(wi * K for wi, K in zip(w, K_tr))
        Kc_te = sum(wi * K for wi, K in zip(w, K_te))
        ev = np.linalg.eigvalsh(Kc_tr)
        if ev.min() < 0:
            Kc_tr += (abs(ev.min()) + 1e-10) * np.eye(Kc_tr.shape[0])

        svm = SVC(kernel='precomputed', C=C, probability=True)
        svm.fit(Kc_tr, y_tr)
        all_auc.append(roc_auc_score(y_te, svm.predict_proba(Kc_te)[:, 1]))

    return {
        'mean': float(np.mean(all_auc)),
        'std':  float(np.std(all_auc, ddof=1)),
        'scores': all_auc,
        'mean_weights': np.mean(all_weights, axis=0),
        'bo_histories': bo_histories,
    }

In [ ]:
# ── Run all methods on all datasets ───────────────────────────────────────────
METHODS_SIMPLE = {
    'Single-Best': weight_fn_single_best,
    'Average':     weight_fn_average,
    'Centered':    weight_fn_centered,
    'SDP':         weight_fn_sdp,
    'Projection':  weight_fn_projection,
}

ALL_RESULTS = {}  # {dataset: {method: result_dict}}
BO_HISTORIES = {} # {dataset: list of convergence histories}

for ds_name in DATASETS:
    print(f'\n{"="*60}')
    print(f'Dataset: {ds_name}')
    print(f'{"="*60}')
    K_list = KERNELS[ds_name]
    y = DATASETS[ds_name]['y']
    ALL_RESULTS[ds_name] = {}

    for method_name, wfn in METHODS_SIMPLE.items():
        t0 = time.time()
        res = evaluate_method(K_list, y, wfn, n_folds=5, seed=SEED)
        ALL_RESULTS[ds_name][method_name] = res
        print(f'  {method_name:15s}: AUC = {res["mean"]:.4f} ± {res["std"]:.4f}  ({time.time()-t0:.1f}s)')

    # Bayesian Optimization
    t0 = time.time()
    bo_res = evaluate_bo(K_list, y, n_folds=5, n_calls=50, seed=SEED)
    ALL_RESULTS[ds_name]['BO'] = {
        'mean': bo_res['mean'], 'std': bo_res['std'],
        'scores': bo_res['scores'], 'mean_weights': bo_res['mean_weights'],
    }
    BO_HISTORIES[ds_name] = bo_res['bo_histories']
    print(f'  {"BO":15s}: AUC = {bo_res["mean"]:.4f} ± {bo_res["std"]:.4f}  ({time.time()-t0:.1f}s)')

print('\nDone!')

In [ ]:
# ── FIGURE 3 : Barres groupées (6 méthodes × 3 datasets) ─────────────────────
plot_method_comparison_grouped(
    ALL_RESULTS,
    title='Figure 3 — Comparaison des 6 méthodes MKL (ROC-AUC)',
    save_path='../results/06/F03_method_comparison.png'
)

In [ ]:
# ── FIGURE 4 : Radar chart multi-métriques ───────────────────────────────────
# Re-evaluate with all metrics for the best dataset
radar_data = {}
ds_name_radar = 'synthetic'
K_list = KERNELS[ds_name_radar]
y = DATASETS[ds_name_radar]['y']

for method_name, wfn in METHODS_SIMPLE.items():
    res = evaluate_method(K_list, y, wfn, n_folds=5, seed=SEED)
    radar_data[method_name] = res['all_metrics']

# Add BO with full metrics
bo_full = evaluate_method(K_list, y,
    lambda K_tr, y_tr: BayesianKernelOptimizer(n_calls=30, random_state=SEED).optimize(K_tr, y_tr)[0],
    n_folds=5, seed=SEED)
radar_data['BO'] = bo_full['all_metrics']

# Rename keys for display
radar_display = {}
for method, metrics in radar_data.items():
    radar_display[method] = {
        'AUC': metrics['auc'], 'Accuracy': metrics['acc'],
        'F1': metrics['f1'], 'Precision': metrics['prec'], 'Recall': metrics['recall']
    }

plot_radar_chart(
    radar_display,
    title=f'Figure 4 — Comparaison multi-métriques ({ds_name_radar})',
    save_path='../results/06/F04_radar_chart.png'
)

## §4 — Deep Dive Bayesian Optimization

In [ ]:
# ── FIGURE 5 : Convergence BO par dataset ─────────────────────────────────────
# Use the first fold's history from each dataset
conv_histories = {ds: BO_HISTORIES[ds][0] for ds in BO_HISTORIES}

plot_bo_convergence(
    conv_histories,
    title='Figure 5 — Convergence BO-MKQSVM (fold 1)',
    save_path='../results/06/F05_bo_convergence.png'
)

In [ ]:
# ── FIGURE 6 : Sensibilité BO — n_calls vs AUC ───────────────────────────────
N_CALLS_RANGE = [15, 30, 50, 80]
sensitivity_results = {}

ds_name_sens = 'synthetic'
K_list = KERNELS[ds_name_sens]
y = DATASETS[ds_name_sens]['y']

# Use a single train/test split for speed
idx = np.arange(len(y))
idx_tr, idx_te = train_test_split(idx, test_size=0.33, random_state=SEED, stratify=y)
y_tr, y_te = y[idx_tr], y[idx_te]
K_tr = [K[np.ix_(idx_tr, idx_tr)] for K in K_list]
K_te = [K[np.ix_(idx_te, idx_tr)] for K in K_list]

for nc in N_CALLS_RANGE:
    print(f'  n_calls={nc}...')
    bo = BayesianKernelOptimizer(n_calls=nc, n_initial_points=min(10, nc//2),
                                 random_state=SEED)
    w, C = bo.optimize(K_tr, y_tr, scoring='accuracy')
    Kc_tr = sum(wi * K for wi, K in zip(w, K_tr))
    Kc_te = sum(wi * K for wi, K in zip(w, K_te))
    ev = np.linalg.eigvalsh(Kc_tr)
    if ev.min() < 0:
        Kc_tr += (abs(ev.min()) + 1e-10) * np.eye(Kc_tr.shape[0])
    svm = SVC(kernel='precomputed', C=C, probability=True)
    svm.fit(Kc_tr, y_tr)
    auc = roc_auc_score(y_te, svm.predict_proba(Kc_te)[:, 1])
    sensitivity_results[nc] = {'auc': auc, 'best_score': bo.get_convergence_info()['best_score'],
                                'history': bo.get_convergence_history()}
    print(f'    AUC={auc:.4f}, best_cv={bo.get_convergence_info()["best_score"]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ncs = sorted(sensitivity_results.keys())
aucs = [sensitivity_results[nc]['auc'] for nc in ncs]
ax.plot(ncs, aucs, 'o-', color='#e74c3c', linewidth=2.5, markersize=10)
ax.set_xlabel('Nombre d\'itérations BO (n_calls)', fontsize=12)
ax.set_ylabel('ROC-AUC test', fontsize=12)
ax.set_title('AUC vs budget BO')
ax.grid(True, alpha=0.3)

ax = axes[1]
colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(ncs)))
for nc, c in zip(ncs, colors):
    h = sensitivity_results[nc]['history']
    ax.plot(h['best_so_far'], '-', color=c, linewidth=2, label=f'n_calls={nc}')
ax.set_xlabel('Itération', fontsize=12)
ax.set_ylabel('Meilleur score CV cumulé', fontsize=12)
ax.set_title('Courbes de convergence comparées')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(f'Figure 6 — Sensibilité BO : n_calls ({ds_name_sens})', fontsize=13)
plt.tight_layout()
plt.savefig('../results/06/F06_bo_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── FIGURE 7 : Poids BO vs Centered (scatter) ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, ds_name in zip(axes, DATASETS):
    w_bo = ALL_RESULTS[ds_name]['BO']['mean_weights']
    w_ca = ALL_RESULTS[ds_name]['Centered']['mean_weights']
    ax.scatter(w_ca, w_bo, c=range(len(KNAMES)), cmap='tab20', s=100,
               edgecolors='black', linewidth=0.5, zorder=3)
    for i, name in enumerate(KNAMES):
        ax.annotate(name[:10], (w_ca[i], w_bo[i]),
                    textcoords='offset points', xytext=(4, 4), fontsize=6)
    lim = max(max(w_bo), max(w_ca)) * 1.1 + 0.01
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, label='y=x')
    ax.set_xlabel('Poids Centered', fontsize=10)
    ax.set_ylabel('Poids BO', fontsize=10)
    ax.set_title(ds_name)
    ax.grid(True, alpha=0.3)

plt.suptitle('Figure 7 — Poids BO vs Centered Alignment', fontsize=13)
plt.tight_layout()
plt.savefig('../results/06/F07_bo_vs_centered_weights.png', dpi=150, bbox_inches='tight')
plt.show()

## §5 — Scaling Study : Performance vs n_qubits

In [ ]:
# ── FIGURE 8 : Scaling n_qubits avec CI95 ────────────────────────────────────
QUBIT_RANGE = [2, 4, 6, 8, 10, 12]
SCALING_METHODS = ['BO', 'Centered', 'Single-Best']
scaling_results = {m: {} for m in SCALING_METHODS}

ds_name_scale = 'synthetic'
X_raw = DATASETS[ds_name_scale]['X_raw']
y = DATASETS[ds_name_scale]['y']

for nq in QUBIT_RANGE:
    print(f'\nn_qubits = {nq}')
    # Preprocess
    reducer = FeatureReducer(n_components=nq)
    scaler  = QuantumScaler(feature_range=(0, 2))
    X_proc = scaler.fit_transform(reducer.fit_transform(X_raw))

    # Build kernels (use standard library for speed at high qubit counts)
    fm_lib_nq = get_extended_feature_map_library(nq)
    knames_nq = list(fm_lib_nq.keys())
    qk_list = [build_quantum_kernel(fm, kernel_type='fidelity') for fm in fm_lib_nq.values()]
    K_list = compute_kernel_matrix_parallel(
        qk_list, X_proc,
        kernel_names=[f'scale_{ds_name_scale}_{nq}q_{n}' for n in knames_nq],
        n_jobs=-1
    )

    # Evaluate methods
    for method in SCALING_METHODS:
        t0 = time.time()
        if method == 'BO':
            bo_res = evaluate_bo(K_list, y, n_folds=3, n_calls=30, seed=SEED)
            scaling_results[method][nq] = {
                'mean': bo_res['mean'], 'std': bo_res['std'], 'scores': bo_res['scores']
            }
        elif method == 'Centered':
            res = evaluate_method(K_list, y, weight_fn_centered, n_folds=3, seed=SEED)
            scaling_results[method][nq] = res
        elif method == 'Single-Best':
            res = evaluate_method(K_list, y, weight_fn_single_best, n_folds=3, seed=SEED)
            scaling_results[method][nq] = res
        print(f'  {method:15s}: {scaling_results[method][nq]["mean"]:.4f} ({time.time()-t0:.0f}s)')

plot_scaling_curve(
    scaling_results,
    title=f'Figure 8 — Performance vs n_qubits (dataset: {ds_name_scale})',
    save_path='../results/06/F08_scaling_qubits.png'
)

In [ ]:
# ── FIGURE 9 : Heatmap concentration (n_qubits × kernel) ─────────────────────
# Collect concentration stats for each (n_qubits, kernel)
conc_data = np.zeros((len(QUBIT_RANGE), len(KNAMES)))

for i, nq in enumerate(QUBIT_RANGE):
    reducer = FeatureReducer(n_components=nq)
    scaler  = QuantumScaler(feature_range=(0, 2))
    X_proc = scaler.fit_transform(reducer.fit_transform(X_raw))
    fm_lib_nq = get_extended_feature_map_library(nq)
    knames_nq = list(fm_lib_nq.keys())
    qk_list = [build_quantum_kernel(fm) for fm in fm_lib_nq.values()]
    K_list = compute_kernel_matrix_parallel(
        qk_list, X_proc,
        kernel_names=[f'conc_{nq}q_{n}' for n in knames_nq],
        n_jobs=-1
    )
    for j, K in enumerate(K_list):
        stats = _kernel_off_diagonal_stats(K)
        conc_data[i, j] = stats['std']

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(conc_data, annot=True, fmt='.3f', cmap='RdYlGn',
            xticklabels=KNAMES, yticklabels=[str(nq) for nq in QUBIT_RANGE],
            ax=ax, linewidths=0.5, annot_kws={'fontsize': 7})
ax.set_xlabel('Kernel', fontsize=11)
ax.set_ylabel('n_qubits', fontsize=11)
ax.set_title('Figure 9 — Concentration (off-diag std) : vert=discriminant, rouge=concentré', fontsize=12)
plt.tight_layout()
plt.savefig('../results/06/F09_concentration_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## §6 — Analyse des poids

In [ ]:
# ── FIGURE 10 : Heatmap poids (méthode × kernel) ─────────────────────────────
ds_name_weights = 'synthetic'
weights_dict = {m: ALL_RESULTS[ds_name_weights][m]['mean_weights'] for m in ALL_RESULTS[ds_name_weights]}

plot_weight_heatmap(
    weights_dict, KNAMES,
    title=f'Figure 10 — Poids MKL par méthode ({ds_name_weights})',
    save_path='../results/06/F10_weight_heatmap.png'
)

In [ ]:
# ── FIGURE 11 : Violin plots stabilité des poids (multi-fold) ────────────────
from src.evaluation.ablation import weight_analysis

_, W_matrix = weight_analysis(
    KERNELS[ds_name_weights], DATASETS[ds_name_weights]['y'],
    weight_fn=weight_fn_centered,
    kernel_names=KNAMES,
    n_folds=5, n_rounds=3, random_state=SEED
)

fig, ax = plt.subplots(figsize=(16, 6))
df_w = pd.DataFrame(W_matrix, columns=KNAMES)
df_melt = df_w.melt(var_name='Kernel', value_name='Poids')
sns.violinplot(data=df_melt, x='Kernel', y='Poids', ax=ax,
               palette='Set2', inner='box', cut=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_title(f'Figure 11 — Stabilité des poids Centered (15 évals, {ds_name_weights})', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/06/F11_weight_violin.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── FIGURE 12 : Concentration vs performance individuelle (scatter) ───────────
ds_name_conc = 'synthetic'
K_list = KERNELS[ds_name_conc]
y = DATASETS[ds_name_conc]['y']

# Single-kernel AUC for each
single_aucs = []
stats_list = []
for K in K_list:
    stats_list.append(_kernel_off_diagonal_stats(K))
    # Quick 3-fold CV
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    aucs = []
    for idx_tr, idx_te in skf.split(np.zeros(len(y)), y):
        K_tr = K[np.ix_(idx_tr, idx_tr)]
        K_te = K[np.ix_(idx_te, idx_tr)]
        ev = np.linalg.eigvalsh(K_tr)
        if ev.min() < 0:
            K_tr = K_tr + (abs(ev.min()) + 1e-10) * np.eye(K_tr.shape[0])
        svm = SVC(kernel='precomputed', C=1.0, probability=True)
        svm.fit(K_tr, y[idx_tr])
        aucs.append(roc_auc_score(y[idx_te], svm.predict_proba(K_te)[:, 1]))
    single_aucs.append(np.mean(aucs))

plot_concentration_scatter(
    stats_list, single_aucs, KNAMES,
    title=f'Figure 12 — Concentration vs Performance ({ds_name_conc})',
    save_path='../results/06/F12_concentration_vs_perf.png'
)

## §7 — Fidelity vs Projected Kernel

In [ ]:
# ── FIGURE 13 : Fidelity vs Projected (paired bar) ───────────────────────────
fid_vs_proj = {}

for ds_name in DATASETS:
    X_proc = DATASETS[ds_name]['X']
    y = DATASETS[ds_name]['y']
    fid_vs_proj[ds_name] = {}

    for ktype in ['fidelity', 'projected']:
        print(f'{ds_name} — {ktype}...')
        fm_lib_k = get_extended_feature_map_library(N_QUBITS)
        qk_list = [build_quantum_kernel(fm, kernel_type=ktype, gamma=1.0) for fm in fm_lib_k.values()]
        K_list = compute_kernel_matrix_parallel(
            qk_list, X_proc,
            kernel_names=[f'{ds_name}_{ktype}_{n}' for n in fm_lib_k.keys()],
            n_jobs=-1
        )
        res = evaluate_method(K_list, y, weight_fn_centered, n_folds=5, seed=SEED)
        fid_vs_proj[ds_name][ktype] = res
        print(f'  AUC = {res["mean"]:.4f} ± {res["std"]:.4f}')

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ds_names = list(fid_vs_proj.keys())
x = np.arange(len(ds_names))
w = 0.35
fid_vals = [fid_vs_proj[d]['fidelity']['mean'] for d in ds_names]
fid_errs = [fid_vs_proj[d]['fidelity']['std'] for d in ds_names]
proj_vals = [fid_vs_proj[d]['projected']['mean'] for d in ds_names]
proj_errs = [fid_vs_proj[d]['projected']['std'] for d in ds_names]

ax.bar(x - w/2, fid_vals, w, yerr=fid_errs, label='Fidelity', color='#3498db',
       edgecolor='black', capsize=5)
ax.bar(x + w/2, proj_vals, w, yerr=proj_errs, label='Projected', color='#e67e22',
       edgecolor='black', capsize=5)
ax.set_xticks(x)
ax.set_xticklabels(ds_names, fontsize=11)
ax.set_ylabel('ROC-AUC', fontsize=12)
ax.set_title('Figure 13 — Fidelity vs Projected Quantum Kernel (Centered MKL)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/06/F13_fidelity_vs_projected.png', dpi=150, bbox_inches='tight')
plt.show()

## §8 — Synthèse

In [ ]:
# ── FIGURE 14 : Panel de synthèse 2×3 ────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# (0,0) Method comparison — synthetic
ax = axes[0, 0]
methods_sorted = sorted(ALL_RESULTS['synthetic'].keys(),
                        key=lambda m: ALL_RESULTS['synthetic'][m]['mean'], reverse=True)
vals = [ALL_RESULTS['synthetic'][m]['mean'] for m in methods_sorted]
errs = [ALL_RESULTS['synthetic'][m]['std'] for m in methods_sorted]
colors = ['#e74c3c' if 'BO' in m else '#3498db' for m in methods_sorted]
ax.barh(range(len(methods_sorted)), vals, xerr=errs, color=colors,
        edgecolor='black', linewidth=0.5, capsize=3)
ax.set_yticks(range(len(methods_sorted)))
ax.set_yticklabels(methods_sorted, fontsize=9)
ax.set_xlabel('ROC-AUC')
ax.set_title('A) Méthodes (synthetic)')
ax.grid(axis='x', alpha=0.3)

# (0,1) Scaling curve
ax = axes[0, 1]
for method in scaling_results:
    nqs = sorted(scaling_results[method].keys())
    means = [scaling_results[method][nq]['mean'] for nq in nqs]
    ax.plot(nqs, means, 'o-', linewidth=2, markersize=6, label=method)
ax.set_xlabel('n_qubits')
ax.set_ylabel('ROC-AUC')
ax.set_title('B) Scaling')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (0,2) BO convergence
ax = axes[0, 2]
for ds_name, h in conv_histories.items():
    ax.plot(h['best_so_far'], '-', linewidth=2, label=ds_name)
ax.set_xlabel('Itération BO')
ax.set_ylabel('Best CV score')
ax.set_title('C) Convergence BO')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (1,0) Concentration heatmap (subset)
ax = axes[1, 0]
subset_k = [0, 3, 6, 10, 14, 18]
sns.heatmap(conc_data[:, subset_k], annot=True, fmt='.3f', cmap='RdYlGn',
            xticklabels=[KNAMES[i] for i in subset_k],
            yticklabels=[str(nq) for nq in QUBIT_RANGE],
            ax=ax, linewidths=0.5, annot_kws={'fontsize': 8})
ax.set_title('D) Concentration')

# (1,1) Weight heatmap (subset methods)
ax = axes[1, 1]
W_sub = {m: ALL_RESULTS['synthetic'][m]['mean_weights'] for m in ['BO', 'Centered', 'Single-Best']}
W_arr = np.array([W_sub[m] for m in W_sub])
sns.heatmap(W_arr[:, :10], annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=KNAMES[:10], yticklabels=list(W_sub.keys()),
            ax=ax, linewidths=0.5, annot_kws={'fontsize': 8})
ax.set_title('E) Poids (10 premiers kernels)')

# (1,2) Fidelity vs Projected
ax = axes[1, 2]
ds_list = list(fid_vs_proj.keys())
x = np.arange(len(ds_list))
ax.bar(x - 0.17, [fid_vs_proj[d]['fidelity']['mean'] for d in ds_list], 0.34,
       label='Fidelity', color='#3498db', edgecolor='black')
ax.bar(x + 0.17, [fid_vs_proj[d]['projected']['mean'] for d in ds_list], 0.34,
       label='Projected', color='#e67e22', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(ds_list, fontsize=9)
ax.set_ylabel('ROC-AUC')
ax.set_title('F) Fidelity vs Projected')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Figure 14 — Synthèse de l\'étude QMKL complète', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('../results/06/F14_synthesis_panel.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Tableau récapitulatif final ───────────────────────────────────────────────
print('\n' + '='*80)
print('TABLEAU RÉCAPITULATIF — ROC-AUC (mean ± std)')
print('='*80)

rows = []
for ds_name in ALL_RESULTS:
    for method in ALL_RESULTS[ds_name]:
        r = ALL_RESULTS[ds_name][method]
        rows.append({'Dataset': ds_name, 'Method': method,
                     'AUC_mean': r['mean'], 'AUC_std': r['std']})

df_summary = pd.DataFrame(rows)
df_pivot = df_summary.pivot(index='Method', columns='Dataset', values='AUC_mean')
df_pivot_std = df_summary.pivot(index='Method', columns='Dataset', values='AUC_std')

# Format
display_df = df_pivot.copy()
for col in display_df.columns:
    display_df[col] = [f'{m:.3f}±{s:.3f}' for m, s in
                        zip(df_pivot[col], df_pivot_std[col])]

print(display_df.to_string())

# Best method per dataset
print('\nMeilleure méthode par dataset :')
for col in df_pivot.columns:
    best = df_pivot[col].idxmax()
    print(f'  {col}: {best} ({df_pivot.loc[best, col]:.4f})')

print('\n✅ Étude complète terminée ! 14 figures sauvées dans results/06/')